# ReplyCoach V6 — MiniLM cross-encoder fine-tuning

Private GPU experiment. Train: 190 reviewed conversations / 1,140 pairs. Validation: 50 conversations / 300 pairs. Mismatched replies are weak negatives, not human-negative gold. No test items or AI judging labels are attached. Fixed seed 425, AdamW 2e-5, batch16, maximum four epochs. Select the best checkpoint by validation pair loss; compare reply quality separately after training. Outputs contain model weights.


In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "transformers==5.19.0", "tokenizers==0.23.2", "safetensors==0.8.0"])


In [ ]:
from pathlib import Path
import json, os, torch
assert torch.cuda.is_available(), "Select a Kaggle GPU before running production training"
print(json.dumps({"torch": torch.__version__, "gpu": torch.cuda.get_device_name(0)}))
bundles = [p.parent for p in Path("/kaggle/input").rglob("manifest.json") if (p.parent/"train_pairs.jsonl").is_file() and (p.parent/"train_reranker.py").is_file()]
assert len(bundles) == 1, "Attach exactly one private V6 training dataset"
bundle = bundles[0]
output = Path("/kaggle/working/reranker_v6")
assert not output.exists(), "Use a fresh saved run; do not overwrite an earlier checkpoint"


In [ ]:
training_env = dict(os.environ, OMP_NUM_THREADS="2", MKL_NUM_THREADS="2", PYTHONUNBUFFERED="1")
subprocess.check_call([sys.executable, str(bundle/"train_reranker.py"), "--bundle", str(bundle), "--output", str(output)], env=training_env)


In [ ]:
import shutil
result = json.loads((output/"best/v6_training.json").read_text())
assert result["completed"] and result["production_checkpoint"]
assert result["device"] == "cuda" and result["weights_changed"] and result["optimizer_steps"] > 0
archive = shutil.make_archive("/kaggle/working/replycoach_v6_checkpoint", "zip", output)
print(json.dumps({"checkpoint": archive, "best_epoch": result["best_epoch"], "validation_pair_loss": result["best_validation_loss"], "optimizer_steps": result["optimizer_steps"]}))
